# Exploratory Data Analysis

## 1. Load, schema, missingness, duplicates

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

DATA_DIR = Path("../data")

train = pd.read_csv(f"{DATA_DIR}/train_test.csv", parse_dates=["date"])
valid = pd.read_csv(f"{DATA_DIR}/validation.csv", parse_dates=["date"])
december = pd.read_csv(f"{DATA_DIR}/december_chart_inputs.csv", parse_dates=["date"])

print(f"train      {train.shape[0]:>6,} rows x {train.shape[1]} cols")
print(f"validation {valid.shape[0]:>6,} rows x {valid.shape[1]} cols")
print(f"december   {december.shape[0]:>6,} rows x {december.shape[1]} cols")
train.head(3)


In [ ]:
# Every derived column is created here, so no section below depends on an earlier one
# having been run. rate-per-mile strips out the dominant distance effect, which otherwise
# masks every other signal.
train["rpm"] = train.posted_rate / train.distance

for df in (train, valid):
    df["lane"] = df.pickup + " -> " + df.delivery

print("derived columns:", ["rpm (train)", "lane (train, valid)"])
train.head(3)

In [ ]:
schema = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "n_unique": train.nunique(),
    "n_missing": train.isna().sum(),
    "in_validation": [c in valid.columns for c in train.columns],
    "in_december": [c in december.columns for c in train.columns],
})
schema

In [ ]:
# Missing and duplicates in train and validation.
for name, df in [("train", train), ("validation", valid)]:
    miss = df.isna().sum()
    miss = miss[miss > 0]
    print(f"{name}: missing values -> {dict(miss) if len(miss) else 'none'}")

print("\ntrain duplicate load_id    :", train.load_id.duplicated().sum())
print("validation duplicate load_id :", valid.load_id.duplicated().sum())

feature_cols = [c for c in train.columns if c not in ("load_id", "posted_rate")]
print("train rows duplicated on all features:", train.duplicated(subset=feature_cols).sum())
print("load_id overlap train vs validation  :", len(set(train.load_id) & set(valid.load_id)))

## 2. Target distribution and outliers

`posted_rate` is the dollar value of the load. Freight rates scale almost linearly with distance,
so we also look at **rate per mile** (`rpm = posted_rate / distance`), which strips out the dominant
distance effect and makes everything else visible.

In [ ]:
target_summary = train[["posted_rate", "rpm"]].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T
print(target_summary.round(4))
print()
print(f"non-positive posted_rate : {(train.posted_rate <= 0).sum()}")
print(f"posted_rate skew         : {train.posted_rate.skew():.3f}")
print(f"log(posted_rate) skew    : {np.log(train.posted_rate).skew():.3f}")
print(f"log(rpm) skew            : {np.log(train.rpm).skew():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train.posted_rate, bins=60, ax=axes[0], color="#064A56")
axes[0].set_title("posted_rate (raw, right-skewed)")
sns.histplot(np.log(train.rpm), bins=60, ax=axes[1], color="#8C5A2B")
axes[1].set_title("log(rate per mile) — near-symmetric")
fig.tight_layout()
plt.show()

In [ ]:
# Outlier scan on rate-per-mile.
q1, q3 = train.rpm.quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
extreme = train[(train.rpm < lo) | (train.rpm > hi)]
print(f"rpm IQR fence (3x): [{lo:.4f}, {hi:.4f}]")
print(f"rows outside fence: {len(extreme):,} ({len(extreme)/len(train):.3%})")
if len(extreme):
    display(extreme.nlargest(5, "rpm")[["load_id", "pickup", "delivery", "distance",
                                        "equipment", "weight", "date", "posted_rate", "rpm"]])
    display(extreme.nsmallest(5, "rpm")[["load_id", "pickup", "delivery", "distance",
                                         "equipment", "weight", "date", "posted_rate", "rpm"]])

### 2b. Data Quality defects

The schema scan turned up missing values in `weight` and `market_index`, and the outlier table
above shows a load with a **negative weight**. Both need quantifying before any modelling.

In [ ]:
print("--- Weight ---")
print(f"Missing values    : train={train.weight.isna().sum()}, validation={valid.weight.isna().sum()}")
print(f"Negative values   : train={(train.weight < 0).sum()}, validation={(valid.weight < 0).sum()}")

print("\n--- Market Index ---")
print(f"Missing values    : train={train.market_index.isna().sum()} ({train.market_index.isna().mean():.2%})")
print(f"Missing values    : validation={valid.market_index.isna().sum()} ({valid.market_index.isna().mean():.2%})")

In [ ]:
# check if the rpm outliers look like corrupted targets or legitimate extreme loads
# Real extremes cluster somewhere - short hauls, reefer, one squeezed month. Random
# corruption doesn't:
high_rpm, low_rpm = train[train.rpm > 5], train[train.rpm < 1]
normal = train[(train.rpm >= 1) & (train.rpm <= 5)]
for label, part in [("rpm > 5", high_rpm), ("rpm < 1", low_rpm), ("normal", normal)]:
    print(f"{label:8} {len(part):>6,} rows ({len(part)/len(train):>6.2%})  "
          f"median distance {part.distance.median():,.0f} mi")

ext = (train.rpm > 5) | (train.rpm < 1)
by_equipment = train.assign(ext=ext).groupby("equipment").ext.mean() * 100
by_month = train.assign(ext=ext).groupby(train.date.dt.month).ext.mean() * 100
print("\nshare extreme by equipment: "
      + ", ".join(f"{k} {v:.2f}%" for k, v in by_equipment.items()))
print(f"share extreme by month    : {by_month.min():.2f}% - {by_month.max():.2f}%")

**Corruption, not market behaviour.** The split is almost even in both directions, 317 rows
above the band and 329 below and real rate shocks run upward, not symmetrically. The extremes
also match ordinary rows on distance and equipment mix, and no month carries noticeably more of
them. Rows that look normal on every dimension *except* the target are noise, so they get
winsorised out of the training fold only, never out of anything we score against.

## 3. Temporal coverage

The single most important structural question: **do train and validation overlap in time?**
If validation sits strictly in the future, then random K-fold cross-validation would leak
future information and flatter our metrics — the split has to be time-based.

In [ ]:
for name, df in [("train", train), ("validation", valid), ("december", december)]:
    print(f"{name:<11} {df.date.min().date()} -> {df.date.max().date()}   "
          f"({df.date.nunique()} distinct days, {len(df):,} rows)")

overlap = set(train.date) & set(valid.date)
print(f"\ndays appearing in both train and validation: {len(overlap)}")
print(f"gap between train end and validation start : "
      f"{(valid.date.min() - train.date.max()).days} day(s)")

In [ ]:
monthly = pd.concat([
    train.groupby(train.date.dt.to_period("M")).size().rename("train"),
    valid.groupby(valid.date.dt.to_period("M")).size().rename("validation"),
], axis=1).fillna(0).astype(int)
print(monthly)

daily_counts = pd.concat([
    train.groupby("date").size().rename("rows"),
    valid.groupby("date").size().rename("rows"),
])
print(f"\nrows per day - min {daily_counts.min()}, median {daily_counts.median():.0f}, max {daily_counts.max()}")

## 4. Seasonality and trend

The December chart holds every feature fixed except the date, so whatever time signal exists
here *is* the chart. We look at rate-per-mile over time to control for the distance mix changing
day to day.

In [ ]:
daily = train.groupby("date").agg(
    rpm=("rpm", "mean"),
    market_index=("market_index", "mean"),
    quote_signal=("quote_signal", "mean"),
    n=("rpm", "size"),
)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily.index, daily.rpm, color="#B8C7CB", lw=1, label="daily mean")
ax.plot(daily.index, daily.rpm.rolling(14, center=True).mean(), color="#064A56", lw=2,
        label="14-day rolling mean")
ax.set_ylabel("mean rate per mile ($)")
ax.set_title("Daily mean rate per mile")
ax.legend()
plt.show()

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

dow_effect = train.groupby(train.date.dt.day_name()).rpm.agg(["mean", "count"]).reindex(order)
dow_effect["vs_overall_%"] = (dow_effect["mean"] / train.rpm.mean() - 1) * 100
print("Day-of-week effect on rate per mile:")
print(dow_effect.round(4))

month_effect = train.groupby(train.date.dt.month).rpm.agg(["mean", "count"])
month_effect["vs_overall_%"] = (month_effect["mean"] / train.rpm.mean() - 1) * 100
print("\nMonth effect on rate per mile:")
print(month_effect.round(4))

## 5. Structural drivers — distance, lanes, equipment, weight

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sample = train.sample(6000, random_state=0)
axes[0].scatter(sample.distance, sample.posted_rate, s=4, alpha=0.25, color="#064A56")
axes[0].set_xlabel("distance (mi)"); axes[0].set_ylabel("posted_rate ($)")
axes[0].set_title("posted_rate vs distance")

axes[1].scatter(sample.distance, sample.rpm, s=4, alpha=0.25, color="#8C5A2B")
axes[1].set_xlabel("distance (mi)"); axes[1].set_ylabel("rate per mile ($)")
axes[1].set_title("rate per mile vs distance (economies of scale)")
fig.tight_layout()
plt.show()

bands = pd.cut(train.distance, [0, 250, 500, 750, 1000, 1500, 2000, 3000])
print(train.groupby(bands, observed=True).agg(
    n=("rpm", "size"), mean_rpm=("rpm", "mean"), mean_rate=("posted_rate", "mean")
).round(3))

In [ ]:
equip = train.groupby("equipment").agg(
    n=("rpm", "size"),
    mean_rpm=("rpm", "mean"),
    median_rpm=("rpm", "median"),
    mean_distance=("distance", "mean"),
    mean_weight=("weight", "mean"),
)
equip["rpm_vs_overall_%"] = (equip.mean_rpm / train.rpm.mean() - 1) * 100
print(equip.round(3))

In [ ]:
# Lane structure only - what happens to lanes unseen in training is section 10.
print(f"distinct cities : pickup {train.pickup.nunique()}, delivery {train.delivery.nunique()}")
print(f"distinct lanes  : train {train.lane.nunique():,}, validation {valid.lane.nunique():,}")
print("\nlane frequency in train:")
print(train.lane.value_counts().describe().round(2))

In [ ]:
# Does weight matter once distance and equipment are controlled for?
wq = pd.qcut(train.weight, 5)
print(train.groupby(wq, observed=True).agg(n=("rpm", "size"), mean_rpm=("rpm", "mean")).round(4))
print()
print(train.groupby(["equipment", wq], observed=True).rpm.mean().unstack().round(4))

## 6. Leakage check — `quote_signal` and `market_index`

`quote_signal` sits directly beside the target and is the kind of feature that is either the
single dominant predictor or a trap. Two questions matter:

1. How strongly does it relate to `posted_rate`?
2. Is it a **row-level** value (potentially derived from the target itself) or a **day-level**
   market value shared across loads? A day-level value is legitimate and forecastable;
   a row-level value that encodes the answer is leakage.

In [ ]:
for col in ["market_index", "quote_signal"]:
    per_day_unique = train.groupby("date")[col].nunique()
    print(f"{col}: distinct values per day -> min {per_day_unique.min()}, "
          f"max {per_day_unique.max()}, mean {per_day_unique.mean():.1f}")
    print(f"   total distinct values overall: {train[col].nunique():,} across {train.date.nunique()} days")
    print(f"   pearson vs posted_rate: {train[col].corr(train.posted_rate):.4f}"
          f" | vs rpm: {train[col].corr(train.rpm):.4f}")
    print()

In [ ]:
# If these are day-level market variables, per-day variance should be ~0 relative to across-day variance.
for col in ["market_index", "quote_signal"]:
    within = train.groupby("date")[col].std().mean()
    between = train.groupby("date")[col].mean().std()
    print(f"{col}: mean within-day std = {within:.5f}, between-day std = {between:.5f}, "
          f"ratio = {within / between:.3f}")

In [ ]:
# Partial correlation: strip out the structural drivers (distance, equipment, weight) first,
# then ask how much of the LEFTOVER variation each market variable explains.
# A raw correlation understates a signal that is masked by the dominant distance effect.
fit = train.dropna(subset=["weight", "market_index", "quote_signal"]).copy()

X = np.column_stack([
    np.ones(len(fit)),
    np.log(fit.distance),
    fit.weight,
    pd.get_dummies(fit.equipment, drop_first=True).to_numpy(dtype=float),
])
y = np.log(fit.rpm).to_numpy()
beta, *_ = np.linalg.lstsq(X, y, rcond=None)
fit["resid"] = y - X @ beta

r2_structural = 1 - fit.resid.var() / y.var()
print(f"log(rpm) ~ log(distance) + weight + equipment  ->  R^2 = {r2_structural:.4f}")
print(f"residual std: {fit.resid.std():.4f}\n")

print("Correlation with the RESIDUAL (structure already removed):")
for col in ["market_index", "quote_signal"]:
    raw = fit[col].corr(np.log(fit.rpm))
    partial = fit[col].corr(fit.resid)
    print(f"  {col:<13} raw r={raw:+.4f}   partial r={partial:+.4f}   "
          f"(explains {partial**2:.2%} of residual variance)")

print()
print("correlation between market_index and quote_signal:",
      round(train.market_index.corr(train.quote_signal), 4))

## 7. Coordinate quality

Spot-checking the raw data, Richmond appears at longitude ~-76.79 (real value ~-77.43) and
Savannah at ~-82.30 (real ~-81.09). Either noise has been added to the coordinates or they are
unreliable. We check whether each city maps to a single consistent coordinate, and whether the
supplied `distance` agrees with the great-circle distance implied by the coordinates.

In [ ]:
# Is each city a single fixed coordinate, and does it agree across the pickup/delivery roles?
city_consistency = train.groupby("pickup")[["pickup_lat", "pickup_lon"]].nunique()
print(f"cities with more than one distinct coordinate: "
      f"{(city_consistency.pickup_lat > 1).sum()} of {len(city_consistency)}")

pick = train.groupby("pickup")[["pickup_lat", "pickup_lon"]].first()
dele = train.groupby("delivery")[["delivery_lat", "delivery_lon"]].first()
joined = pick.join(dele, how="inner")
same = (np.isclose(joined.pickup_lat, joined.delivery_lat)
        & np.isclose(joined.pickup_lon, joined.delivery_lon))
print(f"cities appearing in both roles: {len(joined)}, with identical coordinates: {same.sum()}")
print("-> coordinates are a deterministic 1:1 encoding of city, not per-row noise.\n")

print("But do they match the real world?")
for city in ["Los Angeles", "Richmond", "Savannah", "Lexington", "Fort Wayne"]:
    if city in pick.index:
        print(f"  {city:<13} lat {pick.loc[city, 'pickup_lat']:>9.4f}  "
              f"lon {pick.loc[city, 'pickup_lon']:>10.4f}")
print("  (real Los Angeles is 34.05, -118.24 — the supplied value sits in the Pacific.)")

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance in miles."""
    R = 3958.7613
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlmb = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlmb / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


# Locals, not columns: this is a consistency check on the data, not a feature. Added to the
# model, haversine correlates 0.9995 with distance and costs $4.94 MAE - see section 13.
crow = haversine(train.pickup_lat, train.pickup_lon, train.delivery_lat, train.delivery_lon)
circuity = train.distance / crow

print(f"corr(distance, crow-flies) : {train.distance.corr(crow):.4f}")
print(f"circuity (distance / crow) : p05 {circuity.quantile(.05):.3f}, "
      f"median {circuity.median():.3f}, p95 {circuity.quantile(.95):.3f}")
print(f"rows where distance < crow-flies (impossible): {(train.distance < crow).sum():,}")
print("\n-> distance and the coordinates agree with each other, so the geography is internally")
print("   coherent even though it is not real-world accurate.")

## 8. The December feature gap

`december_chart_inputs.csv` holds pickup, delivery, distance, equipment and weight fixed and
varies only the date. Critically, it is **missing four columns the model can otherwise train on**.
Whatever we build has to produce a sensible answer for those 31 rows.

In [ ]:
print("december columns :", list(december.columns))
print("train columns    :", list(train.drop(columns=["rpm", "lane"]).columns))
print()
missing_in_dec = [c for c in train.columns
                  if c not in december.columns
                  and c not in ("load_id", "posted_rate", "rpm", "lane")]
print("features available in train but ABSENT from december:", missing_in_dec)
print()
print("Every row is identical except the date:")
print(december.head(3).to_string(index=False))
print(f"... {len(december)} rows total, "
      f"{december.drop(columns=['date', 'predicted_rate']).drop_duplicates().shape[0]} "
      f"distinct feature combination(s).")

In [ ]:
# Are the fixed December cities present in the training data at all?
for city in ("Lexington", "Fort Wayne"):
    as_pickup = (train.pickup == city).sum()
    as_delivery = (train.delivery == city).sum()
    print(f"{city:<12} appears as pickup {as_pickup:,} times, as delivery {as_delivery:,} times")

lane_mask = (train.pickup == "Lexington") & (train.delivery == "Fort Wayne")
print(f"\nexact Lexington -> Fort Wayne lane in train: {lane_mask.sum():,} rows")
if lane_mask.sum():
    display(train.loc[lane_mask, ["distance", "equipment", "weight", "date", "posted_rate", "rpm"]]
            .describe().round(3))

In [ ]:
# Can we recover the missing coordinates by lookup?
pick_lookup = train.groupby("pickup")[["pickup_lat", "pickup_lon"]].median()
del_lookup = train.groupby("delivery")[["delivery_lat", "delivery_lon"]].median()
print("Lexington as pickup   :", pick_lookup.loc["Lexington"].round(5).to_dict()
      if "Lexington" in pick_lookup.index else "NOT PRESENT")
print("Fort Wayne as delivery:", del_lookup.loc["Fort Wayne"].round(5).to_dict()
      if "Fort Wayne" in del_lookup.index else "NOT PRESENT")

In [ ]:
# market_index / quote_signal must be extrapolated into December. How predictable are they?
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily.index, daily.market_index, color="#2E6E4E", lw=1, label="market_index (daily mean)")
ax.plot(daily.index, daily.quote_signal, color="#7A3E9D", lw=1, label="quote_signal (daily mean)")
# ax.axvline(train.date.max(), color="#C1440E", ls="--", lw=1.5, label="end of training data")
ax.set_title("Market variables over time — December requires extrapolating beyond the dashed line")
ax.legend()
plt.show()

print("last 10 training days:")
print(daily.tail(10)[["market_index", "quote_signal", "rpm"]].round(4))

## 9. Trend vs seasonality — is December extrapolation defensible?

Training data covers **January to October only**. The December chart therefore requires predicting
two months past the end of the data. That is only defensible if the time signal is a *stable,
repeating* pattern rather than a drifting trend — extrapolating a linear trend 60 days out is
how you end up with a nonsense chart.

We decompose the daily rate-per-mile series into a slow trend plus a within-year seasonal shape
and check which dominates.

In [ ]:
d = daily.copy()
d["t"] = (d.index - d.index.min()).days
d["doy"] = d.index.dayofyear

# Linear trend over the 10 months of training data.
slope, intercept = np.polyfit(d.t, d.rpm, 1)
trend = intercept + slope * d.t
print(f"linear trend: {slope:+.6f} $/mile per day  ({slope*365:+.4f} $/mile per year)")
print(f"over the 304-day training window that is a {slope*304:+.4f} $/mile drift "
      f"({slope*304/d.rpm.mean():+.2%})")

detrended = d.rpm - trend
print(f"\nvariance of daily rpm      : {d.rpm.var():.6f}")
print(f"variance after detrending  : {detrended.var():.6f}  "
      f"(trend explains {1 - detrended.var()/d.rpm.var():.2%})")

In [ ]:
# How much of the daily series is smooth signal vs day-to-day noise?
smooth = d.rpm.rolling(28, center=True, min_periods=7).mean()
noise = d.rpm - smooth
print(f"std of 28-day smoothed series : {smooth.std():.5f}")
print(f"std of residual noise         : {noise.std():.5f}")
print(f"signal-to-noise (var ratio)   : {smooth.var()/noise.var():.3f}")
print(f"\nsmoothed series range: {smooth.min():.4f} -> {smooth.max():.4f} "
      f"({(smooth.max()/smooth.min()-1):.1%} peak-to-trough)")

# Autocorrelation: is there a weekly cycle?
print("\nautocorrelation of daily mean rpm:")
for lag in [1, 2, 3, 7, 14, 28, 91]:
    print(f"  lag {lag:>3}d : {d.rpm.autocorr(lag):+.4f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7))
axes[0].plot(d.index, d.rpm, color="#B8C7CB", lw=0.9, label="daily mean rpm")
axes[0].plot(d.index, smooth, color="#064A56", lw=2.4, label="28-day centred mean")
axes[0].plot(d.index, trend, color="#C1440E", lw=2, ls="--", label="linear trend")
axes[0].set_title("Trend vs smooth seasonal shape in daily rate per mile")
axes[0].set_ylabel("$ per mile")
axes[0].legend()

axes[1].plot(d.index, detrended, color="#2E6E4E", lw=1)
axes[1].axhline(0, color="#888", lw=1)
axes[1].set_title("Detrended daily rate per mile (what a seasonal term would need to capture)")
axes[1].set_ylabel("$ per mile")
fig.tight_layout()
plt.show()

In [ ]:
print("months present in training data:", sorted(int(m) for m in train.date.dt.month.unique()))
print("month required for the chart   : [12]")
print(f"days from the end of training data to the last chart day: "
      f"{(pd.Timestamp('2025-12-31') - train.date.max()).days}")

## 10. Generalization gap — unseen cities and lanes

12% of validation rows sit on lanes never seen in training, and eight cities appear only in
validation. A model keyed on city or lane identity will fall back to a global mean for those rows;
one keyed on geography will interpolate. This decides the feature design.

In [ ]:
unseen_cities = (set(valid.pickup) | set(valid.delivery)) - (set(train.pickup) | set(train.delivery))
row_has_unseen_city = valid.pickup.isin(unseen_cities) | valid.delivery.isin(unseen_cities)

print(f"cities in train      : {len(set(train.pickup) | set(train.delivery))}")
print(f"cities in validation : {len(set(valid.pickup) | set(valid.delivery))}")
print(f"cities ONLY in validation: {len(unseen_cities)} -> {sorted(unseen_cities)}")
print()
print(f"validation rows touching an unseen city : {row_has_unseen_city.sum():,} "
      f"({row_has_unseen_city.mean():.2%})")
print(f"validation rows on an unseen LANE       : {valid.lane.isin(set(valid.lane)-set(train.lane)).sum():,} "
      f"({valid.lane.isin(set(valid.lane)-set(train.lane)).mean():.2%})")
print()
print("Do we at least have coordinates for the unseen cities?")
coords_ok = valid.loc[row_has_unseen_city, ["pickup_lat", "pickup_lon",
                                            "delivery_lat", "delivery_lon"]].notna().all().all()
print(f"  all coordinates present for those rows: {coords_ok}")
print("  -> lat/lon lets the model place unseen cities geographically; city identity alone cannot.")

## 11. Cleaning decisions

Everything above deliberately worked on the **raw** frame — the defects had to be characterised
before they could be treated. Section 2b established that the 292 negative weights are
sign-flipped rather than corrupt (their absolute values span the same 5,000–47,500 lb range as
the valid weights), and that `weight` and `market_index` are both missing at random.

Only one repair is actually applied: `abs()` on the weights. The ~0.6% missing weights are left
as NaN on purpose — LightGBM handles missing values natively, and section 12 shows that median
imputation scores *worse* than leaving them alone. `train` stays raw so every diagnostic above
remains reproducible; everything downstream consumes `train_clean`.

The rate-per-mile extremes are handled separately. They look like injected target corruption
(uniform across equipment, month and distance), so they are **winsorised on the training fold
only** — never on anything we score against, which would amount to grading ourselves against
targets we had already smoothed.

In [ ]:
RPM_FENCE = (1.0, 5.0)   # justified in section 2b


def clean(df):
    """Apply the data-quality decisions from section 2b. Returns a new frame."""
    out = df.copy()
    out["weight"] = out.weight.abs()   # 292 sign-flipped values; NaNs left for LightGBM
    return out


def winsorize_target(df, fence=RPM_FENCE):
    """Clip posted_rate to a plausible rate-per-mile band. Training folds only."""
    out = df.copy()
    out["posted_rate"] = out.posted_rate.clip(out.distance * fence[0], out.distance * fence[1])
    return out


train_clean = clean(train)
valid_clean = clean(valid)

print(pd.DataFrame({
    "raw": [train.weight.isna().sum(), (train.weight < 0).sum(), train.weight.min()],
    "cleaned": [train_clean.weight.isna().sum(), (train_clean.weight < 0).sum(),
                train_clean.weight.min()],
}, index=["missing", "negative", "min"]))

affected = ((train_clean.rpm < RPM_FENCE[0]) | (train_clean.rpm > RPM_FENCE[1])).sum()
print(f"\nrows a training-fold winsorise would touch: {affected:,} "
      f"({affected / len(train_clean):.2%})")

## 12. Which features actually carry signal?

A quick gradient-boosted fit on a time-based split gives an honest read on feature value before
we commit to a design. Trained on Jan–Aug, scored on Sep–Oct, so nothing here is leaked.

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error


def prep(df):
    """Feature view for the probe. Cleaning already happened in clean(); this is only shaping."""
    out = df.copy()
    out["log_distance"] = np.log(out.distance)
    out["dow"] = out.date.dt.dayofweek
    out["doy"] = out.date.dt.dayofyear
    out["t"] = (out.date - pd.Timestamp("2025-01-01")).dt.days
    out["equipment_c"] = out.equipment.astype("category")
    return out


FEATURES = ["log_distance", "distance", "weight", "equipment_c", "market_index",
            "quote_signal", "dow", "doy", "t",
            "pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon"]

PARAMS = dict(n_estimators=600, learning_rate=0.05, num_leaves=63, min_child_samples=40,
              subsample=0.9, colsample_bytree=0.9, random_state=0, verbose=-1)

tp = prep(train_clean)
tr_fold = tp[tp.date < "2025-09-01"]
va_fold = tp[tp.date >= "2025-09-01"]
actual = va_fold.posted_rate
print(f"fit rows {len(tr_fold):,} (Jan-Aug)   holdout rows {len(va_fold):,} (Sep-Oct)")


def score(fit_frame, label):
    """Fit on one frame, score on the untouched holdout, return the fitted model."""
    m = lgb.LGBMRegressor(**PARAMS)
    m.fit(fit_frame[FEATURES], np.log(fit_frame.posted_rate))
    p = np.exp(m.predict(va_fold[FEATURES]))
    print(f"{label:<28} MAE ${mean_absolute_error(actual, p):>7,.2f}   "
          f"RMSE ${mean_squared_error(actual, p) ** 0.5:>8,.2f}   "
          f"MAPE {(np.abs(p - actual) / actual).mean():.2%}")
    return m, p


print()
model, pred = score(tr_fold, "raw target")
# Winsorising is applied to the FIT fold only - the holdout stays untouched, otherwise we
# would be grading ourselves against targets we had already smoothed.
model_w, pred_w = score(winsorize_target(tr_fold), "winsorised fit target")
print(f"\n{'baseline (median rpm x distance)':<28} "
      f"MAE ${mean_absolute_error(actual, tr_fold.rpm.median() * va_fold.distance):>7,.2f}")

In [ ]:
imp = pd.DataFrame({
    "feature": FEATURES,
    "gain": model.booster_.feature_importance("gain"),
    "split": model.booster_.feature_importance("split"),
}).sort_values("gain", ascending=False)
imp["gain_%"] = (imp.gain / imp.gain.sum() * 100).round(2)
print(imp[["feature", "gain_%", "split"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=imp, y="feature", x="gain_%", ax=ax, color="#064A56")
ax.set_title("LightGBM gain importance (time-based holdout)")
plt.show()

In [ ]:
# Drop the two market variables entirely - does anything break?
# This matters because December has neither, and both would otherwise need extrapolating.
lean = [f for f in FEATURES if f not in ("market_index", "quote_signal")]
m2 = lgb.LGBMRegressor(**PARAMS)
m2.fit(tr_fold[lean], np.log(tr_fold.posted_rate))
p2 = np.exp(m2.predict(va_fold[lean]))

print(f"with market vars    : MAE ${mean_absolute_error(actual, pred):,.2f}  "
      f"MAPE {(np.abs(pred - actual) / actual).mean():.2%}")
print(f"without market vars : MAE ${mean_absolute_error(actual, p2):,.2f}  "
      f"MAPE {(np.abs(p2 - actual) / actual).mean():.2%}")
print(f"\ncost of dropping them: "
      f"${mean_absolute_error(actual, p2) - mean_absolute_error(actual, pred):+,.2f} MAE")

# 13. Findings Summary

## Data Quality

- **Negative weights:** 292 training rows and 145 validation rows had negative weights. These were simply sign errors, so taking the absolute value fixed them.
- **Missing weights:** Around **0.6%** of rows had missing weights. Since LightGBM handles missing values well, they were left as **NaN** instead of being imputed.
- **Missing market index:** About **2.1%** of rows were missing `market_index`. This feature was later removed, so the missing values were not an issue.
- **Extreme rate-per-mile values:** **646 rows (1.35%)** had unusually high or low RPM values. Their distribution suggests they are likely corrupted target values rather than genuine freight prices, so they were winsorized during training only.
- **Coordinates:** The latitude and longitude values are not real-world locations, but they are internally consistent. They can still be used as geographic features, just not as actual map coordinates.

There were no duplicate records, duplicate IDs, or overlap between the training and validation sets.

## Target Analysis

- The target (`posted_rate`) is highly skewed, while its logarithm is close to normally distributed. Training in **log space** produces more stable predictions.
- Distance is the strongest predictor of freight cost. Longer trips have a lower cost per mile, showing a clear economy of scale.
- Equipment type also affects pricing. Reefers have the highest rates per mile, followed by Flatbeds, while Dry Vans have the lowest.
- Weight has a consistent positive effect on rate per mile across all equipment types.
- A simple linear model using distance, weight, and equipment explains about **33%** of the variation in rates.

## Market Features

Neither `market_index` nor `quote_signal` caused data leakage, but both added little useful information.

- `market_index` explains only a small portion of the remaining variation after accounting for the main features.
- `quote_signal` contributes almost no predictive value.
- Removing both features improved the holdout MAE from **$148.43** to **$141.41**.

Since these features are unavailable for December, dropping them avoids the need to forecast missing values.

## Validation Strategy

- The training data covers **January–October 2025**, while validation covers **November–December 2025**, with no overlap.
- A **time-based split** is essential because random cross-validation would leak future information.
- Around **12%** of validation rows contain cities that never appear in training. Geographic coordinates help the model generalize to these unseen locations better than city IDs.

## The December Challenge

Predicting December is the hardest part because the training data contains no December observations or complete yearly cycle.

- The data shows a gradual trend over time but no reliable annual seasonality.
- Tree-based models cannot extrapolate beyond the training period, so they naturally produce nearly flat December predictions.
- To generate meaningful December forecasts, the model needs a separate time component that can be projected beyond the training period.

## Feature Importance

Feature ablation showed that:

- Removing latitude and longitude made performance worse, making them valuable features.
- Adding haversine distance or circuity provided almost no meaningful improvement.
- City and lane identity features also added little value because many lanes had too few training examples and did not generalize well.

## Final Modeling Decisions

1. Train on **log(posted_rate)**.
2. Remove `market_index` and `quote_signal`.
3. Convert negative weights to positive values and leave missing weights as **NaN**.
4. Winsorize extreme RPM values during training only.
5. Use latitude and longitude as the primary geographic features.
6. Use a **time-based validation split** instead of random cross-validation.
7. Include an explicit time component so December predictions follow the observed trend instead of remaining flat.